In [0]:
# Source

from pyspark.sql.functions import *

df_tx = spark.table("silver.transactions_clean")

df_tx = df_tx.withColumn(
    "transaction_date",
    to_date("date")
)

In [0]:
# Join to dim_date

dim_date = spark.table("gold.dim_date")

df_fact = (
    df_tx
    .join(
        dim_date,
        df_tx.transaction_date == dim_date.date,
        "left"
    )
)

In [0]:
# Join to dim_client

dim_client = spark.table("gold.dim_client")

df_fact = (
    df_fact
    .join(
        dim_client,
        "client_id",
        "left"
    )
)

In [0]:
# Join to dim_merchant
dim_merchant = spark.table("gold.dim_merchant")

df_fact = (
    df_fact
    .join(
        dim_merchant,
        (
            (df_fact.merchant_id == dim_merchant.merchant_id) &
            (df_fact.transaction_date >= dim_merchant.valid_from) &
            (
                (df_fact.transaction_date <= dim_merchant.valid_to) |
                dim_merchant.valid_to.isNull()
            )
        ),
        "left"
    )
)

In [0]:
# Final selection

fact_transactions = (
    df_fact.select(
        col("id").alias("transaction_id"),
        col("date_sk"),
        col("client_sk"),
        col("merchant_sk"),
        col("amount")
    )
)

In [0]:
# Write to table

fact_transactions.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold.fact_transactions")

In [0]:
# Aggregation – daily merchant sales

merchant_daily = (
    fact_transactions
    .groupBy("merchant_sk", "date_sk")
    .agg(
        sum("amount").alias("total_amount"),
        count("*").alias("transactions_count")
    )
)

merchant_daily.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold.agg_merchant_daily")

In [0]:
# Aggregation – client lifetime metrics

client_lifetime = (
    fact_transactions
    .groupBy("client_sk")
    .agg(
        sum("amount").alias("lifetime_value"),
        count("*").alias("total_transactions"),
        max("date_sk").alias("last_transaction_date_sk")
    )
)

client_lifetime.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold.agg_client_lifetime")